In [1]:
# Estas son las funciones que definimos en la actividad anterior.

def map_pairs(records, mapper):
    pairs = []
    for record in records:
        pairs.extend(mapper(record))
    return pairs


def group_by_key(pairs):
    groups = {}
    for key, value in pairs:
        groups.setdefault(key, []).append(value)
    return groups


def reduce_by_key(groups, reducer):
    return [reducer(key, values) for key, values in groups.items()]

In [2]:
# Problema: Mostrar cómo claves reales de telemetría pueden concentrar trabajo y cómo un combiner reduce pares antes del shuffle.

import csv
import gzip
import hashlib
from collections import defaultdict
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data").is_dir() and (p / "submission").is_dir()
)


In [3]:
# Cada fila representa un evento de telemetría de camiones.

with gzip.open(
    ROOT / "data/truck_events.csv.gz", "rt", encoding="utf-8", newline=""
) as file:
    rows = list(csv.DictReader(file))
len(rows), rows[0]["eventKey"], rows[0]["eventType"]


(17075, '14|25|9223370572464814373', 'Normal')

In [4]:
# La misma función decide qué partición recibe cada clave.


def partition(key):
    return int(hashlib.md5(str(key).encode()).hexdigest(), 16) % 4


balanced, skewed = [0] * 4, [0] * 4
for row in rows:
    balanced[partition(row["eventKey"])] += 1
    skewed[partition(row["eventType"])] += 1
balanced, skewed


([4257, 4365, 4248, 4205], [17041, 11, 7, 16])

In [5]:
# El combiner agrega localmente antes de enviar pares al shuffle.

partials = [defaultdict(int) for _ in range(4)]
for row in rows:
    partials[partition(row["eventType"])][row["eventType"]] += 1
combined_pairs = sum(len(item) for item in partials)
len(rows), combined_pairs


(17075, 5)

In [6]:
# La evidencia compara carga de particiones y volumen enviado.

with (ROOT / "submission/partition_loads.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["partition", "event_key_load", "event_type_load"],
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerows(
        {"partition": i, "event_key_load": balanced[i], "event_type_load": skewed[i]}
        for i in range(4)
    )
with (ROOT / "submission/shuffle_comparison.csv").open(
    "w", encoding="utf-8", newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["raw_pairs", "pairs_after_local_combiner", "hot_key_load"],
        lineterminator="\n",
    )
    writer.writeheader()
    writer.writerow(
        {
            "raw_pairs": len(rows),
            "pairs_after_local_combiner": combined_pairs,
            "hot_key_load": max(skewed),
        }
    )
assert combined_pairs < len(rows)
